# 01 — Raw to Bronze
Carga do CSV para a camada Bronze. Dataset: Kobe Bryant Shot Log (NBA, LA Lakers, 1996–2016).

*Co-authored with CoCo*

In [ ]:
%%sql -r ctx
USE WAREHOUSE KOBE_WH;
USE SCHEMA KOBE_DB.BRONZE;

In [ ]:
%%sql -r create_bronze
CREATE OR REPLACE TABLE KOBE_DB.BRONZE.SHOTS_RAW (
    ROW_INDEX           INTEGER,
    MATCH_EVENT_ID      FLOAT,
    LOCATION_X          FLOAT,
    LOCATION_Y          FLOAT,
    REMAINING_MIN       FLOAT,
    POWER_OF_SHOT       FLOAT,
    KNOCKOUT_MATCH      FLOAT,
    GAME_SEASON         VARCHAR(10),
    REMAINING_SEC       FLOAT,
    DISTANCE_OF_SHOT    FLOAT,
    IS_GOAL             FLOAT,
    AREA_OF_SHOT        VARCHAR(50),
    SHOT_BASICS         VARCHAR(50),
    RANGE_OF_SHOT       VARCHAR(50),
    TEAM_NAME           VARCHAR(100),
    DATE_OF_GAME        DATE,
    HOME_AWAY           VARCHAR(50),
    SHOT_ID_NUMBER      FLOAT,
    LATITUDE            FLOAT,
    LONGITUDE           FLOAT,
    TYPE_OF_SHOT        VARCHAR(50),
    TYPE_OF_COMBINED_SHOT VARCHAR(50),
    MATCH_ID            INTEGER,
    TEAM_ID             BIGINT,
    REMAINING_MIN_2     FLOAT,
    POWER_OF_SHOT_2     FLOAT,
    KNOCKOUT_MATCH_2    FLOAT,
    REMAINING_SEC_2     FLOAT,
    DISTANCE_OF_SHOT_2  FLOAT,
    -- Auditoria
    DATA_CARGA          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

In [ ]:
%%sql -r stage_upload
-- Criar stage e copiar CSV do workspace
CREATE OR REPLACE STAGE KOBE_DB.BRONZE.LOAD_STAGE;

COPY FILES INTO @KOBE_DB.BRONZE.LOAD_STAGE
FROM 'snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/'
FILES=('.temp/uploads/yds_data.csv');

In [ ]:
%%sql -r load_bronze
-- Carregar CSV na Bronze (inclui ROW_INDEX, separa lat/lng)
COPY INTO KOBE_DB.BRONZE.SHOTS_RAW (
    ROW_INDEX, MATCH_EVENT_ID, LOCATION_X, LOCATION_Y,
    REMAINING_MIN, POWER_OF_SHOT, KNOCKOUT_MATCH, GAME_SEASON,
    REMAINING_SEC, DISTANCE_OF_SHOT, IS_GOAL, AREA_OF_SHOT,
    SHOT_BASICS, RANGE_OF_SHOT, TEAM_NAME, DATE_OF_GAME,
    HOME_AWAY, SHOT_ID_NUMBER, LATITUDE, LONGITUDE,
    TYPE_OF_SHOT, TYPE_OF_COMBINED_SHOT, MATCH_ID, TEAM_ID,
    REMAINING_MIN_2, POWER_OF_SHOT_2, KNOCKOUT_MATCH_2,
    REMAINING_SEC_2, DISTANCE_OF_SHOT_2
)
FROM (
    SELECT
        $1,   -- row_index
        $2,   -- match_event_id
        $3,   -- location_x
        $4,   -- location_y
        $5,   -- remaining_min
        $6,   -- power_of_shot
        $7,   -- knockout_match
        $8,   -- game_season
        $9,   -- remaining_sec
        $10,  -- distance_of_shot
        $11,  -- is_goal
        $12,  -- area_of_shot
        $13,  -- shot_basics
        $14,  -- range_of_shot
        $15,  -- team_name
        TRY_TO_DATE($16, 'YYYY-MM-DD'),  -- date_of_game
        $17,  -- home/away
        $18,  -- shot_id_number
        TRY_TO_DOUBLE(SPLIT_PART($19, ',', 1)),  -- latitude
        TRY_TO_DOUBLE(SPLIT_PART($19, ',', 2)),  -- longitude
        $20,  -- type_of_shot
        $21,  -- type_of_combined_shot
        $22,  -- match_id
        $23,  -- team_id
        $24,  -- remaining_min_2
        $25,  -- power_of_shot_2
        $26,  -- knockout_match_2
        $27,  -- remaining_sec_2
        $28   -- distance_of_shot_2
    FROM @KOBE_DB.BRONZE.LOAD_STAGE/.temp/uploads/yds_data.csv
)
FILE_FORMAT = (TYPE='CSV' SKIP_HEADER=1 FIELD_OPTIONALLY_ENCLOSED_BY='"' EMPTY_FIELD_AS_NULL=TRUE)
ON_ERROR = 'CONTINUE'
FORCE = TRUE;

## Validação

In [ ]:
%%sql -r validation
SELECT 
    COUNT(*) AS TOTAL_LINHAS,
    COUNT(DISTINCT MATCH_ID) AS JOGOS,
    COUNT(DISTINCT GAME_SEASON) AS TEMPORADAS,
    MIN(DATE_OF_GAME) AS PRIMEIRO_JOGO,
    MAX(DATE_OF_GAME) AS ULTIMO_JOGO,
    SUM(CASE WHEN IS_GOAL IS NULL THEN 1 ELSE 0 END) AS LINHAS_SEM_TARGET
FROM KOBE_DB.BRONZE.SHOTS_RAW;

In [ ]:
%%sql -r sample_data
-- Amostra dos dados
SELECT * FROM KOBE_DB.BRONZE.SHOTS_RAW LIMIT 5;